# Exercise 4 — run_backtest

`run_backtest` is the engine that powers every trading experiment in Section 7. It takes an OHLCV DataFrame and a signal Series, applies a one-day lag to prevent look-ahead bias, and returns a dict of metrics plus the equity curve. The one-day lag is the most important detail: you see the signal at the end of today, and trade at the start of tomorrow.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def compute_returns(df):
    return df["Close"].pct_change()

def compute_equity(returns, initial=1.0):
    return (1 + returns.fillna(0)).cumprod() * initial
def max_drawdown(equity):
    peak = equity.cummax()
    return float(((equity - peak) / peak).min())
def sharpe_ratio(returns, periods_per_year=252):
    clean = returns.dropna()
    if len(clean) == 0 or clean.std() == 0:
        return 0.0
    return float(clean.mean() / clean.std() * (periods_per_year ** 0.5))

REQUIRED_KEYS = {
    "total_return", "annualized_return", "sharpe_ratio", "max_drawdown",
    "win_rate", "n_trades", "equity", "strategy_returns", "market_returns",
}

def run_backtest(df, signals):
    """Vectorised daily backtest with look-ahead prevention.

    Args:
        df      : OHLCV DataFrame (needs "Close")
        signals : pd.Series aligned to df.index; values in {-1, 0, 1}

    Critical: apply signals.shift(1).fillna(0) before multiplying by returns.
    This means the signal you see at end of day t enters your portfolio on
    day t+1 — no peeking at tomorrow's price.

    Returns a dict with keys: total_return, annualized_return, sharpe_ratio,
    max_drawdown, win_rate, n_trades, equity, strategy_returns, market_returns.
    """
    # TODO:
    # 1. market_returns   = compute_returns(df)
    # 2. positions        = signals.shift(1).fillna(0)
    # 3. strategy_returns = positions * market_returns
    # 4. equity           = compute_equity(strategy_returns)
    # 5. clean            = strategy_returns.dropna(); n_days = len(clean)
    # 6. win_rate         = float((clean > 0).sum() / max(n_days, 1))
    # 7. n_trades         = int((positions.diff().fillna(0) != 0).sum())
    # 8. total_ret        = float(equity.iloc[-1] - 1.0)
    # 9. base = 1 + total_ret; ann_ret = base**(252/max(n_days,1))-1 if base>0 else -1.0
    # 10. return dict with all REQUIRED_KEYS
    n  = len(df)
    eq = pd.Series([1.0] * n, index=df.index)
    mr = compute_returns(df)
    return {
        "total_return":      0.0,
        "annualized_return": 0.0,
        "sharpe_ratio":      0.0,
        "max_drawdown":      0.0,
        "win_rate":          0.0,
        "n_trades":          0,
        "equity":            eq,
        "strategy_returns":  pd.Series([0.0] * n, index=df.index),
        "market_returns":    mr,
    }


### Checks

In [ ]:
checks = 0

# 1 — returns dict with all required keys
try:
    df      = _synthetic()
    signals = pd.Series(1, index=df.index)
    result  = run_backtest(df, signals)
    assert isinstance(result, dict)
    missing = REQUIRED_KEYS - result.keys()
    assert not missing, f"missing keys: {missing}"
    checks += 1; print("✅ 1 run_backtest returns dict with all required keys")
except Exception as e:
    print("❌ 1:", e)

# 2 — equity[-1] == 1 + total_return
try:
    df      = _synthetic()
    signals = pd.Series(1, index=df.index)
    r       = run_backtest(df, signals)
    diff    = abs(r["equity"].iloc[-1] - (1 + r["total_return"]))
    assert diff < 1e-9, f"equity[-1] != 1+total_return: diff={diff}"
    checks += 1; print("✅ 2 equity[-1] == 1 + total_return")
except Exception as e:
    print("❌ 2:", e)

# 3 — always-long: strategy ≈ buy-and-hold (same market returns)
try:
    df      = _synthetic()
    signals = pd.Series(1, index=df.index)
    r       = run_backtest(df, signals)
    # positions = signals.shift(1) = [0, 1, 1, ..., 1]
    # strategy_returns differs from market_returns only on day 0 (position=0)
    sr = r["strategy_returns"].iloc[2:]
    mr = r["market_returns"].iloc[2:]
    diff = (sr - mr).abs().max()
    assert diff < 1e-9, f"always-long should match buy-and-hold after day 1, diff={diff}"
    checks += 1; print("✅ 3 always-long matches buy-and-hold (after warmup day)")
except Exception as e:
    print("❌ 3:", e)

# 4 — look-ahead bias: first strategy_return is NaN or 0 even when signal[0]=1
try:
    df      = _synthetic()
    signals = pd.Series(0, index=df.index)
    signals.iloc[0] = 1   # first day: buy signal
    r = run_backtest(df, signals)
    sr0 = r["strategy_returns"].iloc[0]
    assert pd.isna(sr0) or abs(sr0) < 1e-12,         f"look-ahead bias: strategy_returns[0] should be 0/NaN, got {sr0}"
    # Signal from day 0 should be applied on day 1
    sr1 = r["strategy_returns"].iloc[1]
    mr1 = r["market_returns"].iloc[1]
    assert abs(sr1 - mr1) < 1e-12,         f"signal[0]=1 should give strategy_returns[1]=market_returns[1]"
    checks += 1; print("✅ 4 signals are shifted by 1: no look-ahead bias")
except Exception as e:
    print("❌ 4:", e)

# 5 — flat signals → n_trades ≤ 1 (only the initial entry if any)
try:
    df = _synthetic()
    flat = pd.Series(0, index=df.index)   # never trade
    r   = run_backtest(df, flat)
    assert r["n_trades"] == 0, f"flat signal → 0 trades, got {r['n_trades']}"
    always_long = pd.Series(1, index=df.index)
    r2 = run_backtest(df, always_long)
    assert r2["n_trades"] == 1, f"always-long → 1 entry trade, got {r2['n_trades']}"
    checks += 1; print("✅ 5 n_trades: flat=0, always-long=1")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
